# Hybrid LSTM-CNN for IoT Intrusion Detection
**Replicating Sinha et al., 2025** — *A High Performance Hybrid LSTM-CNN Secure Architecture for IoT Environments Using Deep Learning*

This notebook is fully self-contained. Just hit **Runtime > Run All**.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
%pip install -q numpy pandas scikit-learn scipy xgboost tensorflow keras matplotlib seaborn imbalanced-learn pyyaml kaggle joblib tqdm

In [ ]:
# ============================================================
# CELL 2: Kaggle Credentials & Dataset Download
# ============================================================
import os, glob

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_e0733d21bd2433de364ab020c3f1a0c7'

DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

if not glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
    print('Downloading BoT-IoT dataset from Kaggle...')
    !kaggle datasets download -d majedjaber/bot-iot-all-features-5-sample -p {DATA_DIR} --unzip
else:
    print(f'Dataset already present ({len(glob.glob(os.path.join(DATA_DIR, "**/*.csv"), recursive=True))} CSV files)')

In [ ]:
# ============================================================
# CELL 3: Load and Preprocess Data
# ============================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
print(f'Loading {len(csv_files)} CSV file(s)...')
dfs = [pd.read_csv(f, low_memory=False) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)
print(f'Total records: {len(df):,}')
print(f'Columns: {list(df.columns)}')

SAMPLE_FRAC = 0.1  # Use 10% for a quick test; set to 1.0 for full run
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
    print(f'Sampled to {len(df):,} records ({SAMPLE_FRAC*100:.0f}%)')

In [ ]:
# ============================================================
# CELL 4: Clean and Preprocess
# ============================================================
df = df.replace([np.inf, -np.inf], np.nan)
df = df.drop_duplicates()
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            df[col] = df[col].fillna(df[col].mean())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')
print(f'After cleaning: {len(df):,} records')

target_col = None
for candidate in ['attack', 'Attack', 'label', 'Label', 'category', 'Category']:
    if candidate in df.columns:
        target_col = candidate
        break
if target_col is None:
    print('WARNING: Could not auto-detect target column! Available columns:')
    print(df.columns.tolist())
    target_col = input('Enter the target column name: ')
print(f'Target column: {target_col}')
print(f'Class distribution:\n{df[target_col].value_counts()}')

exclude_cols = {target_col}
for col in df.columns:
    if col.lower() in ['pkseqid', 'saddr', 'daddr', 'sport', 'dport', 'category', 'subcategory']:
        if col != target_col:
            exclude_cols.add(col)

cat_cols = [c for c in df.select_dtypes(include=['object', 'category']).columns if c not in exclude_cols]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
    print(f'One-hot encoded {len(cat_cols)} columns. Total features: {len(df.columns)}')

feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].values.astype(np.float32)

le = LabelEncoder()
y = le.fit_transform(df[target_col])
print(f'Target classes: {dict(zip(le.classes_, range(len(le.classes_))))}')
print(f'Features shape: {X.shape}')

In [ ]:
# ============================================================
# CELL 5: Split, Scale, SMOTE
# ============================================================
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Split: Train={len(X_train):,} | Val={len(X_val):,} | Test={len(X_test):,}')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

pca = PCA(n_components=0.95, random_state=42)
X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
X_test = pca.transform(X_test)
print(f'PCA reduced features to: {X_train.shape[1]}')

print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
smote_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('undersample', RandomUnderSampler(random_state=42))
])
X_train, y_train = smote_pipeline.fit_resample(X_train, y_train)
print(f'After SMOTE:  {dict(pd.Series(y_train).value_counts())}')

X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val_dl = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
X_test_dl = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
print(f'DL input shape: {X_train_dl.shape}')

In [ ]:
# ============================================================
# CELL 6: Build ALL Models
# ============================================================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

INPUT_SHAPE = (X_train_dl.shape[1], X_train_dl.shape[2])

def build_lstm_cnn(input_shape):
        inp = layers.Input(shape=input_shape)
    x = inp
    for i in range(3):
        x = layers.LSTM(256, activation='tanh', return_sequences=True, name=f'lstm_{i+1}')(x)
        x = layers.Dropout(0.2, name=f'lstm_drop_{i+1}')(x)
    for i, f in enumerate([64, 128, 256]):
        x = layers.Conv1D(f, 3, activation='relu', padding='same', name=f'conv_{i+1}')(x)
    x = layers.GlobalMaxPooling1D(name='global_maxpool')(x)
    x = layers.Dense(128, activation='relu', name='dense1')(x)
    x = layers.Dropout(0.2, name='dense_drop')(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = models.Model(inp, out, name='LSTM_CNN_Hybrid')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(inp)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='CNN')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_rnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.SimpleRNN(128, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.SimpleRNN(128)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='RNN')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_lstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.LSTM(128, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(128)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='LSTM')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_bilstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(128))(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='BiLSTM')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_gru(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.GRU(128, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(128)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='GRU')
    m.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    return m

ALL_MODELS = {
    'LSTM-CNN (Hybrid)': build_lstm_cnn,
    'CNN': build_cnn,
    'RNN': build_rnn,
    'LSTM': build_lstm,
    'BiLSTM': build_bilstm,
    'GRU': build_gru,
}

print(f'Defined {len(ALL_MODELS)} models. Input shape: {INPUT_SHAPE}')

In [ ]:
# ============================================================
# CELL 7: Train ALL Models
# ============================================================
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

EPOCHS = 50
BATCH_SIZE = 128

all_results = {}
all_histories = {}
all_predictions = {}
trained_models = {}

for name, builder in ALL_MODELS.items():
    print(f'\n{"="*60}')
    print(f'  Training: {name}')
    print(f'{"="*60}')
    
    model = builder(INPUT_SHAPE)
    model.summary()
    
    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
    ]
    
    start = time.time()
    history = model.fit(
        X_train_dl, y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_data=(X_val_dl, y_val),
        callbacks=cbs, verbose=1
    )
    train_time = time.time() - start
    
    y_proba = model.predict(X_test_dl, verbose=0).flatten()
    y_pred = (y_proba > 0.5).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
    
    # All metrics to 2 decimal places
    metrics = {
        'Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)': round(precision_score(y_test, y_pred, zero_division=0) * 100, 2),
        'Recall (%)': round(recall_score(y_test, y_pred, zero_division=0) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, y_pred, zero_division=0) * 100, 2),
        'FPR (%)': round((fp / (fp + tn) * 100) if (fp + tn) > 0 else 0.0, 2),
        'Detection Rate (%)': round((tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0, 2),
        'AUC-ROC (%)': round(roc_auc_score(y_test, y_proba) * 100, 2),
        'Train Time (s)': round(train_time, 1),
        'TP': int(tp),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'Epochs Run': len(history.history['loss']),
    }
    
    all_results[name] = metrics
    all_histories[name] = history.history
    all_predictions[name] = {'y_true': y_test, 'y_pred': y_pred, 'y_proba': y_proba}
    trained_models[name] = model
    
    print(f'\n  Accuracy:       {metrics["Accuracy (%)"]:.2f}%')
    print(f'  Precision:      {metrics["Precision (%)"]:.2f}%')
    print(f'  Recall:         {metrics["Recall (%)"]:.2f}%')
    print(f'  F1-Score:       {metrics["F1-Score (%)"]:.2f}%')
    print(f'  FPR:            {metrics["FPR (%)"]:.2f}%')
    print(f'  Detection Rate: {metrics["Detection Rate (%)"]:.2f}%')
    print(f'  AUC-ROC:        {metrics["AUC-ROC (%)"]:.2f}%')
    print(f'  TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}')
    print(f'  Epochs: {metrics["Epochs Run"]}  |  Time: {metrics["Train Time (s)"]}s')

print('\n' + '='*60)
print('  ALL MODELS TRAINED!')
print('='*60)

In [ ]:
# ============================================================
# CELL 8: Results Comparison Table
# ============================================================
display_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'FPR (%)', 'Detection Rate (%)', 'AUC-ROC (%)', 'Train Time (s)', 'Epochs Run']
results_df = pd.DataFrame(all_results).T[display_cols]
results_df.index.name = 'Model'

highlight_max_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Detection Rate (%)', 'AUC-ROC (%)']
highlight_min_cols = ['FPR (%)']

print('\n=== MODEL COMPARISON (All values to 2 decimal places) ===')
styled = results_df.style.format('{:.2f}')\
    .highlight_max(axis=0, subset=highlight_max_cols, color='#c6efce')\
    .highlight_min(axis=0, subset=highlight_min_cols, color='#c6efce')\
    .set_caption('Performance Comparison — Sinha et al. (2025) Replication')
display(styled)

In [ ]:
# ============================================================
# CELL 9: Visualizations
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

# --- 9a. Training History (LSTM-CNN) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = all_histories['LSTM-CNN (Hybrid)']
ax1.plot(h['loss'], label='Train', color=COLORS[0], lw=2)
ax1.plot(h['val_loss'], label='Val', color=COLORS[3], lw=2)
ax1.set_title('LSTM-CNN Hybrid - Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.plot(h['accuracy'], label='Train', color=COLORS[1], lw=2)
ax2.plot(h['val_accuracy'], label='Val', color=COLORS[4], lw=2)
ax2.set_title('LSTM-CNN Hybrid - Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

# --- 9b. ROC Curves ---
fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, data) in enumerate(all_predictions.items()):
    fpr_vals, tpr_vals, _ = roc_curve(data['y_true'], data['y_proba'])
    roc_auc = auc(fpr_vals, tpr_vals)
    ax.plot(fpr_vals, tpr_vals, color=COLORS[i%6], lw=2, label=f'{name} (AUC={roc_auc:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models', fontweight='bold')
ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

# --- 9c. Comparison Bar Chart ---
fig, ax = plt.subplots(figsize=(14, 6))
bar_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Detection Rate (%)']
plot_df = results_df[bar_cols]
plot_df.plot(kind='bar', ax=ax, color=COLORS[:5], width=0.75)
ax.set_ylabel('Score (%)', fontsize=12); ax.set_title('Model Comparison — All Metrics', fontweight='bold', fontsize=14)
ax.set_ylim([max(plot_df.min().min()-2, 90), 101])
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=6, padding=2)
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()

# --- 9d. Detailed Confusion Matrices (ALL models) ---
n_models = len(all_predictions)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, data) in enumerate(all_predictions.items()):
    ax = axes[idx]
    cm = confusion_matrix(data['y_true'], data['y_pred'], labels=[0, 1])
    tn_v, fp_v, fn_v, tp_v = cm.ravel()
    total = cm.sum()
    
    # Create annotation with count, percentage, and label
    cm_pct = cm / total * 100
    annot = np.array([
        [f'TN\n{tn_v:,}\n({cm_pct[0,0]:.2f}%)', f'FP\n{fp_v:,}\n({cm_pct[0,1]:.2f}%)'],
        [f'FN\n{fn_v:,}\n({cm_pct[1,0]:.2f}%)', f'TP\n{tp_v:,}\n({cm_pct[1,1]:.2f}%)']
    ])
    
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=True,
                xticklabels=['Normal (0)', 'Attack (1)'],
                yticklabels=['Normal (0)', 'Attack (1)'],
                annot_kws={'size': 11, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)
    
    # Title with key metrics
    m = all_results[name]
    ax.set_title(f'{name}\nAcc={m["Accuracy (%)"]:.2f}%  F1={m["F1-Score (%)"]:.2f}%  FPR={m["FPR (%)"]:.2f}%', 
                 fontweight='bold', fontsize=10)

# Hide extra subplots if fewer than 6 models
for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Detailed Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- 9e. Print full classification reports ---
print('\n' + '='*60)
print('  DETAILED CLASSIFICATION REPORTS')
print('='*60)
for name, data in all_predictions.items():
    print(f'\n--- {name} ---')
    print(classification_report(data['y_true'], data['y_pred'], 
          target_names=['Normal', 'Attack'], digits=4))
    cm = confusion_matrix(data['y_true'], data['y_pred'], labels=[0,1])
    tn_v, fp_v, fn_v, tp_v = cm.ravel()
    print(f'  Total Samples: {cm.sum():,}')
    print(f'  TP={tp_v:,}  TN={tn_v:,}  FP={fp_v:,}  FN={fn_v:,}')
    print(f'  Detection Rate: {all_results[name]["Detection Rate (%)"]:.2f}%')
    print(f'  FPR:            {all_results[name]["FPR (%)"]:.2f}%')

In [ ]:
# ============================================================
# CELL 9b: Learning Curves Analysis (All Models Comparison)
# ============================================================
# Paper 1-style: 4 subplots comparing all models across epochs

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

plot_configs = [
    ('loss',         'Training Loss Comparison',      'Loss',     axes[0, 0]),
    ('val_loss',     'Validation Loss Comparison',    'Loss',     axes[0, 1]),
    ('accuracy',     'Training Accuracy Comparison',  'Accuracy', axes[1, 0]),
    ('val_accuracy', 'Validation Accuracy Comparison','Accuracy', axes[1, 1]),
]

LINE_STYLES = ['-', '--', '-.', ':', '-', '--']
MARKERS = ['o', 's', '^', 'D', 'v', 'P']

for metric_key, title, ylabel, ax in plot_configs:
    for i, (name, hist) in enumerate(all_histories.items()):
        epochs_range = range(1, len(hist[metric_key]) + 1)
        ax.plot(epochs_range, hist[metric_key],
                color=COLORS[i % 6], lw=2,
                linestyle=LINE_STYLES[i % 6],
                marker=MARKERS[i % 6], markersize=4, markevery=max(1, len(hist[metric_key])//10),
                label=name, alpha=0.9)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves — All Models Comparison', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Also print epoch summary
print('\nEpochs trained per model:')
for name, hist in all_histories.items():
    n_ep = len(hist['loss'])
    final_loss = hist['val_loss'][-1]
    final_acc = hist['val_accuracy'][-1]
    print(f'  {name:20s} -> {n_ep:2d} epochs  |  val_loss={final_loss:.4f}  val_acc={final_acc:.4f}')

In [ ]:
# ============================================================
# CELL 9c: Model Complexity vs Performance
# ============================================================
import seaborn as sns

complexities = []
for name, model in trained_models.items():
    params = model.count_params()
    time_s = all_results[name]['Train Time (s)']
    acc = all_results[name]['Accuracy (%)']
    complexities.append({
        'Model': name,
        'Parameters': params,
        'Train Time (s)': time_s,
        'Accuracy (%)': acc
    })

comp_df = pd.DataFrame(complexities)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = sns.color_palette("husl", len(comp_df))

# Chart 1: Complexity vs Training Time
for i, row in comp_df.iterrows():
    ax1.scatter(row['Parameters'], row['Train Time (s)'], color=colors[i], s=200, label=row['Model'], alpha=0.8, edgecolors='k')
    ax1.annotate(row['Model'], (row['Parameters'], row['Train Time (s)']), 
                 xytext=(10, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax1.set_xlabel('Model Complexity (Number of Parameters)', fontsize=12)
ax1.set_ylabel('Total Training Time (seconds)', fontsize=12)
ax1.set_title('Complexity vs. Training Time', fontweight='bold', fontsize=14)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.set_xscale('log') # Log scale helps spread out the parameter counts

# Chart 2: Complexity vs Accuracy
for i, row in comp_df.iterrows():
    ax2.scatter(row['Parameters'], row['Accuracy (%)'], color=colors[i], s=200, label=row['Model'], alpha=0.8, edgecolors='k')
    ax2.annotate(row['Model'], (row['Parameters'], row['Accuracy (%)']), 
                 xytext=(10, -5), textcoords='offset points', fontsize=9, fontweight='bold')

ax2.set_xlabel('Model Complexity (Number of Parameters)', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Complexity vs. Accuracy', fontweight='bold', fontsize=14)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.set_xscale('log')
ax2.set_ylim([max(comp_df['Accuracy (%)'].min() - 0.5, 90), 100.2])

plt.tight_layout()
plt.show()

print('\nModel Complexity Table:')
display(comp_df.sort_values('Parameters').style.format({'Parameters': '{:,}', 'Train Time (s)': '{:.1f}', 'Accuracy (%)': '{:.2f}'}))

In [ ]:
# ============================================================
# CELL 10: Feature Importance (Permutation-based)
# ============================================================
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

print('Computing Permutation Feature Importance (this may take a few minutes)...')

lstm_cnn_model = trained_models['LSTM-CNN (Hybrid)']

class KerasWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
        self.classes_ = np.array([0, 1])
    def fit(self, X, y): return self
    def predict(self, X):
        X_3d = X.reshape((X.shape[0], X.shape[1], 1))
        return (self.model.predict(X_3d, verbose=0).flatten() > 0.5).astype(int)
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

wrapper = KerasWrapper(lstm_cnn_model)

n_samples = min(1000, len(X_test))
X_test_subset = X_test[:n_samples]
y_test_subset = y_test[:n_samples]

result = permutation_importance(
    wrapper, X_test_subset, y_test_subset,
    n_repeats=5, random_state=42, n_jobs=-1, scoring='accuracy'
)

sorted_idx = result.importances_mean.argsort()[::-1][:20]
top_features = [feature_cols[i] if i < len(feature_cols) else f'feature_{i}' for i in sorted_idx]
top_importances = result.importances_mean[sorted_idx]
top_stds = result.importances_std[sorted_idx]

fig_feat, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_importances, xerr=top_stds, color=COLORS[0], alpha=0.8)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features)
ax.invert_yaxis()
ax.set_xlabel('Mean Accuracy Decrease')
ax.set_title('LSTM-CNN Hybrid - Top 20 Feature Importance (Permutation)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 Most Important Features:')
for i in range(min(10, len(top_features))):
    print(f'  {i+1}. {top_features[i]}: {top_importances[i]:.4f} +/- {top_stds[i]:.4f}')

In [ ]:
# ============================================================
# CELL 11: SAVE ALL RESULTS TO DISK
# ============================================================
import json
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'results/run_{timestamp}'
FIGS_DIR = f'{RUN_DIR}/figures'
TABLES_DIR = f'{RUN_DIR}/tables'
MODELS_DIR = f'{RUN_DIR}/models'
HISTORY_DIR = f'{RUN_DIR}/training_history'
for d in [FIGS_DIR, TABLES_DIR, MODELS_DIR, HISTORY_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Saving all results to: {RUN_DIR}/')
print('='*60)

# --- 1. Comparison Table ---
results_df.to_csv(f'{TABLES_DIR}/model_comparison.csv')
results_df.to_latex(f'{TABLES_DIR}/model_comparison.tex', float_format='%.2f')
print(f'[Saved] Comparison table (CSV + LaTeX)')

# --- 2. Full Results JSON ---
run_meta = {
    'timestamp': timestamp,
    'sample_frac': SAMPLE_FRAC,
    'total_records': int(len(df)),
    'n_features': int(X.shape[1]),
    'epochs_max': EPOCHS,
    'batch_size': BATCH_SIZE,
    'train_size': int(len(X_train)),
    'val_size': int(len(X_val)),
    'test_size': int(len(X_test)),
}
full_results = {'run_config': run_meta, 'metrics': {}}
for name, metrics in all_results.items():
    full_results['metrics'][name] = {k: float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v for k, v in metrics.items()}
with open(f'{TABLES_DIR}/all_results.json', 'w') as f:
    json.dump(full_results, f, indent=2)
print(f'[Saved] Full results JSON')

# --- 3. Training Histories ---
for name, hist in all_histories.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    hist_serializable = {k: [float(v) for v in vals] for k, vals in hist.items()}
    with open(f'{HISTORY_DIR}/{safe_name}_history.json', 'w') as f:
        json.dump(hist_serializable, f, indent=2)
print(f'[Saved] Training histories')

# --- 4. Figures ---
# 4a. LSTM-CNN training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = all_histories['LSTM-CNN (Hybrid)']
ax1.plot(h['loss'], label='Train', color=COLORS[0], lw=2)
ax1.plot(h['val_loss'], label='Val', color=COLORS[3], lw=2)
ax1.set_title('LSTM-CNN Hybrid - Loss', fontweight='bold'); ax1.legend()
ax2.plot(h['accuracy'], label='Train', color=COLORS[1], lw=2)
ax2.plot(h['val_accuracy'], label='Val', color=COLORS[4], lw=2)
ax2.set_title('LSTM-CNN Hybrid - Accuracy', fontweight='bold'); ax2.legend()
plt.tight_layout()
fig.savefig(f'{FIGS_DIR}/lstm_cnn_training_history.png', dpi=150, bbox_inches='tight'); plt.close(fig)

# 4b. All model training histories
for name, hist in all_histories.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(hist['loss'], label='Train', lw=2); ax1.plot(hist['val_loss'], label='Val', lw=2)
    ax1.set_title(f'{name} - Loss', fontweight='bold'); ax1.legend()
    ax2.plot(hist['accuracy'], label='Train', lw=2); ax2.plot(hist['val_accuracy'], label='Val', lw=2)
    ax2.set_title(f'{name} - Accuracy', fontweight='bold'); ax2.legend()
    plt.tight_layout()
    fig.savefig(f'{FIGS_DIR}/{safe_name}_history.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] Training history plots')

# 4h. Learning curves comparison (all models)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plot_configs = [
    ('loss', 'Training Loss Comparison', 'Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss Comparison', 'Loss', axes[0, 1]),
    ('accuracy', 'Training Accuracy Comparison', 'Accuracy', axes[1, 0]),
    ('val_accuracy', 'Validation Accuracy Comparison', 'Accuracy', axes[1, 1]),
]
LINE_STYLES = ['-', '--', '-.', ':', '-', '--']
MARKERS = ['o', 's', '^', 'D', 'v', 'P']
for metric_key, title, ylabel, ax in plot_configs:
    for i, (name, hist) in enumerate(all_histories.items()):
        epochs_range = range(1, len(hist[metric_key]) + 1)
        ax.plot(epochs_range, hist[metric_key], color=COLORS[i%6], lw=2,
                linestyle=LINE_STYLES[i%6], marker=MARKERS[i%6], markersize=4,
                markevery=max(1, len(hist[metric_key])//10), label=name, alpha=0.9)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9, loc='best'); ax.grid(True, alpha=0.3)
plt.suptitle('Learning Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIGS_DIR}/learning_curves_comparison.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] Learning curves comparison')

# 4i. Model Complexity vs Performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for i, row in comp_df.iterrows():
    ax1.scatter(row['Parameters'], row['Train Time (s)'], color=colors[i], s=200, alpha=0.8, edgecolors='k')
    ax1.annotate(row['Model'], (row['Parameters'], row['Train Time (s)']), xytext=(10, 5), textcoords='offset points', fontsize=9)
ax1.set_xlabel('Model Complexity (Parameters)'); ax1.set_ylabel('Training Time (s)')
ax1.set_title('Complexity vs. Training Time', fontweight='bold')
ax1.set_xscale('log'); ax1.grid(True, linestyle='--', alpha=0.6)

for i, row in comp_df.iterrows():
    ax2.scatter(row['Parameters'], row['Accuracy (%)'], color=colors[i], s=200, alpha=0.8, edgecolors='k')
    ax2.annotate(row['Model'], (row['Parameters'], row['Accuracy (%)']), xytext=(10, -5), textcoords='offset points', fontsize=9)
ax2.set_xlabel('Model Complexity (Parameters)'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Complexity vs. Accuracy', fontweight='bold')
ax2.set_xscale('log'); ax2.grid(True, linestyle='--', alpha=0.6)
ax2.set_ylim([max(comp_df['Accuracy (%)'].min() - 0.5, 90), 100.2])
plt.tight_layout()
fig.savefig(f'{FIGS_DIR}/complexity_analysis.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] Model complexity charts')

# Save complexity table
comp_df.to_csv(f'{TABLES_DIR}/model_complexity.csv', index=False)
print(f'[Saved] Model complexity table')

# 4c. ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, data) in enumerate(all_predictions.items()):
    fpr_vals, tpr_vals, _ = roc_curve(data['y_true'], data['y_proba'])
    roc_auc = auc(fpr_vals, tpr_vals)
    ax.plot(fpr_vals, tpr_vals, color=COLORS[i%6], lw=2, label=f'{name} (AUC={roc_auc:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models', fontweight='bold')
ax.legend(loc='lower right'); plt.tight_layout()
fig.savefig(f'{FIGS_DIR}/roc_curves.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] ROC curves')

# 4d. Comparison bar chart
fig, ax = plt.subplots(figsize=(14, 6))
bar_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Detection Rate (%)']
plot_df = results_df[bar_cols]
plot_df.plot(kind='bar', ax=ax, color=COLORS[:5], width=0.75)
ax.set_ylabel('Score (%)'); ax.set_title('Model Comparison', fontweight='bold')
ax.set_ylim([max(plot_df.min().min()-2, 90), 101])
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=6, padding=2)
plt.tight_layout()
fig.savefig(f'{FIGS_DIR}/comparison_bar.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] Comparison bar chart')

# 4e. Detailed confusion matrices (all models in one image)
n_models = len(all_predictions)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes_flat = axes.flatten()
for idx, (name, data) in enumerate(all_predictions.items()):
    ax = axes_flat[idx]
    cm = confusion_matrix(data['y_true'], data['y_pred'], labels=[0, 1])
    tn_v, fp_v, fn_v, tp_v = cm.ravel()
    total = cm.sum()
    cm_pct = cm / total * 100
    annot = np.array([
        [f'TN\n{tn_v:,}\n({cm_pct[0,0]:.2f}%)', f'FP\n{fp_v:,}\n({cm_pct[0,1]:.2f}%)'],
        [f'FN\n{fn_v:,}\n({cm_pct[1,0]:.2f}%)', f'TP\n{tp_v:,}\n({cm_pct[1,1]:.2f}%)']
    ])
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=True,
                xticklabels=['Normal (0)', 'Attack (1)'], yticklabels=['Normal (0)', 'Attack (1)'],
                annot_kws={'size': 11, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    m = all_results[name]
    ax.set_title(f'{name}\nAcc={m["Accuracy (%)"]:.2f}%  F1={m["F1-Score (%)"]:.2f}%  FPR={m["FPR (%)"]:.2f}%', fontweight='bold', fontsize=9)
for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)
plt.suptitle('Detailed Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIGS_DIR}/confusion_matrices_all.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] Confusion matrices (all models)')

# 4f. Individual confusion matrices
for name, data in all_predictions.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    cm = confusion_matrix(data['y_true'], data['y_pred'], labels=[0, 1])
    tn_v, fp_v, fn_v, tp_v = cm.ravel()
    total = cm.sum()
    cm_pct = cm / total * 100
    annot = np.array([
        [f'TN\n{tn_v:,}\n({cm_pct[0,0]:.2f}%)', f'FP\n{fp_v:,}\n({cm_pct[0,1]:.2f}%)'],
        [f'FN\n{fn_v:,}\n({cm_pct[1,0]:.2f}%)', f'TP\n{tp_v:,}\n({cm_pct[1,1]:.2f}%)']
    ])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=True,
                xticklabels=['Normal (0)', 'Attack (1)'], yticklabels=['Normal (0)', 'Attack (1)'],
                annot_kws={'size': 13, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted Label', fontsize=12); ax.set_ylabel('True Label', fontsize=12)
    m = all_results[name]
    ax.set_title(f'{name} — Confusion Matrix\nAcc={m["Accuracy (%)"]:.2f}%  Prec={m["Precision (%)"]:.2f}%  Recall={m["Recall (%)"]:.2f}%  F1={m["F1-Score (%)"]:.2f}%\nFPR={m["FPR (%)"]:.2f}%  DR={m["Detection Rate (%)"]:.2f}%  Total={total:,}', fontweight='bold', fontsize=10)
    plt.tight_layout()
    fig.savefig(f'{FIGS_DIR}/confusion_matrix_{safe_name}.png', dpi=150, bbox_inches='tight'); plt.close(fig)
print(f'[Saved] Individual confusion matrices')

# 4g. Feature importance
fig_feat2, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_importances, xerr=top_stds, color=COLORS[0], alpha=0.8)
ax.set_yticks(range(len(top_features))); ax.set_yticklabels(top_features)
ax.invert_yaxis(); ax.set_xlabel('Mean Accuracy Decrease')
ax.set_title('LSTM-CNN Hybrid - Top 20 Feature Importance', fontweight='bold')
plt.tight_layout()
fig_feat2.savefig(f'{FIGS_DIR}/feature_importance.png', dpi=150, bbox_inches='tight'); plt.close(fig_feat2)
print(f'[Saved] Feature importance plot')

# --- 5. Save Model Weights ---
for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    model.save(f'{MODELS_DIR}/{safe_name}.keras')
print(f'[Saved] Model weights')

# --- 6. Feature importance CSV ---
feat_df = pd.DataFrame({'Feature': top_features, 'Importance': top_importances, 'Std': top_stds})
feat_df.to_csv(f'{TABLES_DIR}/feature_importance.csv', index=False)

# --- 7. Classification reports ---
with open(f'{TABLES_DIR}/classification_reports.txt', 'w') as f:
    for name, data in all_predictions.items():
        f.write(f'\n{"="*60}\n  {name}\n{"="*60}\n')
        f.write(classification_report(data['y_true'], data['y_pred'], target_names=['Normal', 'Attack'], digits=4))
        cm = confusion_matrix(data['y_true'], data['y_pred'], labels=[0,1])
        tn_v, fp_v, fn_v, tp_v = cm.ravel()
        f.write(f'\n  TP={tp_v:,}  TN={tn_v:,}  FP={fp_v:,}  FN={fn_v:,}\n')
        f.write(f'  Detection Rate: {all_results[name]["Detection Rate (%)"]:.2f}%\n')
        f.write(f'  FPR: {all_results[name]["FPR (%)"]:.2f}%\n')
print(f'[Saved] Classification reports')

print(f'\n{"="*60}')
print(f'  ALL RESULTS SAVED TO: {RUN_DIR}/')
print(f'{"="*60}')
print(f'\nSaved files:')
for root, dirs, files in os.walk(RUN_DIR):
    level = root.replace(RUN_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f'{subindent}{file} ({size_kb:.1f} KB)')

## Summary
The Hybrid LSTM-CNN model combines temporal (LSTM) and spatial (CNN) feature extraction to achieve superior detection of intrusion attacks in IoT environments, replicating the findings of Sinha et al. (2025).

**Next steps**: Integrate Quantum ML (VQC/QAE) into the LSTM-CNN pipeline to explore performance improvements.